## Lab Assignment: Wisconsin Breast Cancer Dataset

### 1. Load the data and classify the variables

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# The CSV is stored in JS02, one directory above this notebook.
df = pd.read_csv("../JS02/wbc.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df["diagnosis"].value_counts())

### 2. Usable and unusable variables; encode the diagnosis

In [ ]:

unusable_columns = ["id", "diagnosis"]
feature_columns = [column for column in df.columns if column not in unusable_columns]

X = df[feature_columns]
y = df["diagnosis"].map({"B": 0, "M": 1}).astype(int)

print("Unusable / non-predictor columns:", unusable_columns)
print("Usable numeric feature count:", len(feature_columns))
print("Usable features:", feature_columns)
print("Encoded target values:", sorted(y.unique().tolist()), "(B=0, M=1)")

### 3. Standardization, feature selection, and Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

def make_pipeline(k):
    return Pipeline([
        ("standardize", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ])

In [ ]:
from sklearn.model_selection import cross_val_score

cv_results = []
for k in range(1, len(feature_columns) + 1):
    candidate = make_pipeline(k)
    scores = cross_val_score(candidate, X_train, y_train, cv=5, scoring="accuracy")
    cv_results.append({"k": k, "cv_mean": scores.mean(), "cv_std": scores.std()})

cv_results = pd.DataFrame(cv_results)
best_row = cv_results.sort_values(["cv_mean", "k"], ascending=[False, True]).iloc[0]
best_k = int(best_row["k"])

print(cv_results.to_string(index=False, formatters={"cv_mean": "{:.4f}".format, "cv_std": "{:.4f}".format}))
print(f"\nOptimal number of features: {best_k}")
print(f"Best mean CV accuracy: {best_row['cv_mean']:.4f}")

### 4. Final test and selected features

In [ ]:
final_pipeline = make_pipeline(best_k)
final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

print("Test accuracy:", f"{accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Benign (B)", "Malignant (M)"]))

selector = final_pipeline.named_steps["select"]
selected_features = [
    feature for feature, selected in zip(feature_columns, selector.get_support()) if selected
]
selected_scores = selector.scores_[selector.get_support()]
selected_summary = pd.DataFrame({
    "feature": selected_features,
    "f_score": selected_scores,
}).sort_values("f_score", ascending=False)

print("Selected features:")
print(selected_summary.to_string(index=False, formatters={"f_score": "{:.4f}".format}))